# HOLa-style predicate description low-rank decomposition

This notebook encodes one text description per scene-graph predicate with the repository-local CLIP text encoder, then computes a HOLa-style low-rank factorization:

$$F \approx W M$$

- `F`: CLIP text features, shape `[num_predicates, clip_dim]`
- `W`: predicate-specific weights, shape `[num_predicates, rank]`
- `M`: shared low-rank basis, shape `[rank, clip_dim]`
- `rank`: low-rank dimension `m`

If a predicate has no LLM description yet, it falls back to `a photo of {predicate}` and records that in metadata.

## 1. Configuration

Edit this cell before running. For a quick smoke test without the VG dict, set `PREDICATE_JSON = None` and use `PREDICATE_LIST`.

In [ ]:
import os

REPO_ROOT = "/Users/shangfei/Developer/SDSGG"

# Use either PREDICATE_JSON or PREDICATE_LIST.
PREDICATE_JSON = os.path.join(REPO_ROOT, "datasets/vg/VG-SGG-dicts-with-attri.json")
PREDICATE_LIST = None  # Example: ["on", "under", "riding", "holding"]

# Optional LLM descriptions. Supported formats:
# JSON: {"on": "Objects are in contact...", "riding": "..."}
# CSV: columns must be predicate,description
DESCRIPTIONS = None

CLIP_MODEL = "ViT-B/32"  # Or a local checkpoint path, e.g. "/path/to/ViT-B-32.pt"
DEVICE = "cuda"  # Change to "cpu" if needed.
RANK = 16
BATCH_SIZE = 64
NORMALIZE_FEATURES = True
KEEP_BACKGROUND = False
PROMPT_TEMPLATE = "a photo of {predicate}"

OUTPUT = os.path.join(REPO_ROOT, "outputs/hola_predicate_factors.pt")

# If your environment is missing CLIP tokenizer dependencies, run this once:
# %pip install ftfy regex tqdm

## 2. Imports and helper functions

In [ ]:
import csv
import json
import sys
from typing import Dict, List, Optional, Sequence, Tuple

import torch
import torch.nn.functional as F

LOCAL_CLIP_PARENT = os.path.join(
    REPO_ROOT, "maskrcnn_benchmark", "modeling", "roi_heads", "relation_head"
)
if LOCAL_CLIP_PARENT not in sys.path:
    sys.path.insert(0, LOCAL_CLIP_PARENT)

print("Repo root:", REPO_ROOT)
print("Local CLIP parent:", LOCAL_CLIP_PARENT)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
def normalize_key(text: str) -> str:
    return " ".join(str(text).strip().lower().split())


def load_predicates_from_json(path: str) -> List[Tuple[int, str]]:
    with open(path, "r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if "idx_to_predicate" in payload:
        idx_to_predicate = payload["idx_to_predicate"]
        if isinstance(idx_to_predicate, dict):
            return sorted(
                ((int(index), name) for index, name in idx_to_predicate.items()),
                key=lambda item: item[0],
            )
        return [(index, name) for index, name in enumerate(idx_to_predicate)]

    if "predicate_to_idx" in payload:
        predicate_to_idx = payload["predicate_to_idx"]
        return sorted(
            ((int(index), name) for name, index in predicate_to_idx.items()),
            key=lambda item: item[0],
        )

    raise ValueError(
        f"No predicate mapping found in {path}. Expected idx_to_predicate or predicate_to_idx."
    )


def load_predicates_from_list(predicate_list: Sequence[str]) -> List[Tuple[int, str]]:
    predicates = [item.strip() for item in predicate_list if item and item.strip()]
    if not predicates:
        raise ValueError("PREDICATE_LIST is empty after parsing.")
    return list(enumerate(predicates))


def filter_background(predicates: Sequence[Tuple[int, str]], keep_background: bool) -> List[Tuple[int, str]]:
    if keep_background:
        return list(predicates)
    return [
        (index, name)
        for index, name in predicates
        if normalize_key(name) != "__background__"
    ]

In [ ]:
def load_description_map(path: str) -> Dict[str, str]:
    ext = os.path.splitext(path)[1].lower()

    if ext == ".json":
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
        if not isinstance(payload, dict):
            raise ValueError("Description JSON must map predicate -> description.")
        return {normalize_key(key): str(value).strip() for key, value in payload.items()}

    if ext == ".csv":
        description_map = {}
        with open(path, "r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            required_fields = {"predicate", "description"}
            if not reader.fieldnames or not required_fields.issubset(reader.fieldnames):
                raise ValueError("Description CSV must contain predicate and description columns.")
            for row in reader:
                predicate = normalize_key(row["predicate"])
                description = row["description"].strip()
                if predicate:
                    description_map[predicate] = description
        return description_map

    raise ValueError(f"Unsupported description format: {path}. Use JSON or CSV.")


def resolve_descriptions(
    predicates: Sequence[Tuple[int, str]],
    description_map: Optional[Dict[str, str]],
    prompt_template: str,
) -> Tuple[List[str], List[bool]]:
    descriptions = []
    fallback_mask = []

    for _, predicate_name in predicates:
        description = ""
        if description_map is not None:
            description = description_map.get(normalize_key(predicate_name), "").strip()

        used_fallback = not description
        if used_fallback:
            description = prompt_template.format(predicate=predicate_name)

        descriptions.append(description)
        fallback_mask.append(used_fallback)

    return descriptions, fallback_mask

In [ ]:
def encode_text_features(
    texts: Sequence[str],
    clip_model_name: str,
    device: str,
    batch_size: int,
) -> torch.Tensor:
    try:
        from CLIP import clip
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError(
            "Failed to import local CLIP dependencies. "
            "Install the missing package in this notebook kernel, e.g. `%pip install ftfy regex tqdm`. "
            f"Missing package: {error.name}"
        ) from error

    try:
        model, _ = clip.load(clip_model_name, device=device)
    except RuntimeError as error:
        raise RuntimeError(
            "Failed to load CLIP model. If weights are not cached locally, pass a local checkpoint "
            "path in CLIP_MODEL or prepare ~/.cache/clip first. "
            f"Original error: {error}"
        ) from error

    model.eval()
    features = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch_texts = list(texts[start:start + batch_size])
            tokens = clip.tokenize(batch_texts).to(device)
            batch_features = model.encode_text(tokens).float().cpu()
            features.append(batch_features)

    return torch.cat(features, dim=0)


def effective_rank(requested_rank: int, features: torch.Tensor) -> int:
    if requested_rank < 1:
        raise ValueError("RANK must be positive.")
    return min(requested_rank, features.shape[0], features.shape[1])


def compute_low_rank_factors(features: torch.Tensor, requested_rank: int):
    u, singular_values, vh = torch.linalg.svd(features, full_matrices=False)
    rank = effective_rank(requested_rank, features)

    truncated_u = u[:, :rank]
    truncated_s = singular_values[:rank]
    truncated_vh = vh[:rank, :]

    W = truncated_u * truncated_s.unsqueeze(0)
    M = truncated_vh
    reconstructed = W @ M

    squared = singular_values.pow(2)
    explained_variance_ratio = squared / squared.sum().clamp_min(1e-12)
    reconstruction_error = torch.norm(features - reconstructed, p="fro")

    return {
        "W": W,
        "M": M,
        "reconstructed": reconstructed,
        "rank": rank,
        "singular_values": singular_values,
        "explained_variance_ratio": explained_variance_ratio,
        "reconstruction_error": reconstruction_error,
    }

## 3. Load predicates and descriptions

In [ ]:
if PREDICATE_JSON and PREDICATE_LIST:
    raise ValueError("Use only one of PREDICATE_JSON or PREDICATE_LIST.")
if not PREDICATE_JSON and not PREDICATE_LIST:
    raise ValueError("Set either PREDICATE_JSON or PREDICATE_LIST.")

if PREDICATE_JSON:
    predicates = load_predicates_from_json(PREDICATE_JSON)
else:
    predicates = load_predicates_from_list(PREDICATE_LIST)

predicates = filter_background(predicates, KEEP_BACKGROUND)
if not predicates:
    raise ValueError("No predicates remain after background filtering.")

description_map = load_description_map(DESCRIPTIONS) if DESCRIPTIONS else None
descriptions, fallback_mask = resolve_descriptions(predicates, description_map, PROMPT_TEMPLATE)

predicate_indices = [index for index, _ in predicates]
predicate_names = [name for _, name in predicates]

print(f"Loaded predicates: {len(predicate_names)}")
print(f"Fallback descriptions: {sum(fallback_mask)}")
print("First predicates:", predicate_names[:10])
print("First descriptions:", descriptions[:3])

## 4. Encode text with CLIP

In [ ]:
features = encode_text_features(
    texts=descriptions,
    clip_model_name=CLIP_MODEL,
    device=DEVICE,
    batch_size=BATCH_SIZE,
)

if NORMALIZE_FEATURES:
    features = F.normalize(features, dim=-1)

print("features:", tuple(features.shape), features.dtype)
print("feature norm range:", float(features.norm(dim=-1).min()), float(features.norm(dim=-1).max()))

## 5. Compute HOLa-style low-rank factors

In [ ]:
factor_payload = compute_low_rank_factors(features, RANK)

W = factor_payload["W"]
M = factor_payload["M"]
reconstructed = factor_payload["reconstructed"]
rank = factor_payload["rank"]

print("Requested rank:", RANK)
print("Effective rank:", rank)
print("W:", tuple(W.shape))
print("M:", tuple(M.shape))
print("reconstructed:", tuple(reconstructed.shape))
print("reconstruction_error:", float(factor_payload["reconstruction_error"]))
print("first singular values:", factor_payload["singular_values"][:10].tolist())

## 6. Save `.pt` factors and `.json` metadata

In [ ]:
def metadata_path(output_path: str) -> str:
    root, _ = os.path.splitext(output_path)
    return root + ".json"


os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)

output_payload = {
    "features": features,
    "W": W,
    "M": M,
    "reconstructed": reconstructed,
    "predicate_names": predicate_names,
    "predicate_indices": predicate_indices,
    "descriptions": descriptions,
    "rank": rank,
    "singular_values": factor_payload["singular_values"],
    "explained_variance_ratio": factor_payload["explained_variance_ratio"],
    "reconstruction_error": float(factor_payload["reconstruction_error"]),
}

metadata = {
    "predicate_json": PREDICATE_JSON,
    "descriptions": DESCRIPTIONS,
    "clip_model": CLIP_MODEL,
    "device": DEVICE,
    "normalize": NORMALIZE_FEATURES,
    "requested_rank": RANK,
    "effective_rank": rank,
    "num_predicates": len(predicate_names),
    "feature_dim": int(features.shape[1]),
    "fallback_count": int(sum(fallback_mask)),
    "fallback_mask": fallback_mask,
    "predicate_names": predicate_names,
    "predicate_indices": predicate_indices,
    "description_texts": descriptions,
    "reconstruction_error": float(factor_payload["reconstruction_error"]),
    "singular_values": factor_payload["singular_values"].tolist(),
    "explained_variance_ratio": factor_payload["explained_variance_ratio"].tolist(),
}

torch.save(output_payload, OUTPUT)
META_OUTPUT = metadata_path(OUTPUT)
with open(META_OUTPUT, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2)

print("Saved factors:", OUTPUT)
print("Saved metadata:", META_OUTPUT)

## 7. Quick inspection

Use this cell to inspect the rows that fell back to predicate-name prompts.

In [ ]:
fallback_examples = [
    (idx, name, desc)
    for idx, name, desc, used_fallback in zip(predicate_indices, predicate_names, descriptions, fallback_mask)
    if used_fallback
]

print(f"Fallback rows: {len(fallback_examples)}")
fallback_examples[:10]